In [1]:
import os,sys
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/V-Max_Hackerz/
sys.path.insert(0,'/content/drive/MyDrive/V-Max_Hackerz/notebook')

Mounted at /content/drive
/content/drive/MyDrive/V-Max_Hackerz


In [3]:
from functools import lru_cache
import re
import numpy as np
import pandas as pd
!pip install rapidfuzz
from rapidfuzz.fuzz import ratio, token_set_ratio, token_sort_ratio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 17.8 MB/s eta 0:00:00


In [7]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/V-Max_Hackerz")
DATA_DIR = PROJECT_ROOT / "student_resource"/"dataset" / "train"

s1 = pd.read_csv(DATA_DIR / "train_source1.tsv", sep="\t", dtype=str)
s2 = pd.read_csv(DATA_DIR / "train_source2.tsv", sep="\t", dtype=str)
s3 = pd.read_csv(DATA_DIR / "train_source3.tsv", sep="\t", dtype=str)
gt = pd.read_csv(DATA_DIR / "train_ground_truth.tsv", sep="\t", dtype=str)

print("S1:", s1.shape)
print("S2:", s2.shape)
print("S3:", s3.shape)
print("GT:", gt.shape)

S1: (2206821, 4)
S2: (5034616, 4)
S3: (5285603, 4)
GT: (2206821, 2)


In [8]:
ground_truth = {}

for row in gt.itertuples(index=False):
    if pd.isna(row.matched_entity_ids) or not row.matched_entity_ids:
        ground_truth[row.source1_entity_id] = set()
    else:
        ground_truth[row.source1_entity_id] = set(
            row.matched_entity_ids.split(",")
        )

print("Ground-truth S1 records:", len(ground_truth))

Ground-truth S1 records: 2206821


In [9]:
from sklearn.model_selection import train_test_split

train_s1_ids, val_s1_ids = train_test_split(
    s1["entity_id"],
    test_size=0.20,
    random_state=42
)

train_s1_ids = set(train_s1_ids)
val_s1_ids = set(val_s1_ids)

print("Training S1:", len(train_s1_ids))
print("Validation S1:", len(val_s1_ids))

Training S1: 1765456
Validation S1: 441365


In [10]:
candidate_source = pd.concat(
    [
        s2[["entity_id", "business_name", "business_address", "country"]],
        s3[["entity_id", "business_name", "business_address", "country"]]
    ],
    ignore_index=True
).set_index("entity_id")

print("Candidate entities:", len(candidate_source))

Candidate entities: 10320219


In [11]:
def normalize_unicode(text):
    if pd.isna(text):
        return ""
    return str(text).casefold()


def normalize_text(text):
    text = normalize_unicode(text)
    if not text:
        return ""

    text = text.replace("&", " and ")
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()


NAME_ABBREVIATIONS = {
    "pvt": "private",
    "ltd": "limited",
    "corp": "corporation",
    "inc": "incorporated",
    "co": "company"
}

ADDRESS_ABBREVIATIONS = {
    "rd": "road",
    "st": "street",
    "ave": "avenue",
    "blvd": "boulevard",
    "dr": "drive",
    "ln": "lane",
    "hwy": "highway",
    "pkwy": "parkway",
    "ct": "court",
    "pl": "place",
    "trl": "trail",
    "cir": "circle",
    "ste": "suite",
    "apt": "apartment"
}


def expand_abbreviations(text, mapping):
    if not text:
        return ""
    return " ".join(mapping.get(token, token) for token in text.split())


def normalize_name(text):
    return expand_abbreviations(
        normalize_text(text),
        NAME_ABBREVIATIONS
    )


def normalize_address(text):
    return expand_abbreviations(
        normalize_text(text),
        ADDRESS_ABBREVIATIONS
    )

In [12]:
from collections import defaultdict
from itertools import combinations

NAME_STOPWORDS = {
    "private", "limited", "corporation", "company",
    "incorporated", "llc", "llp", "pllc", "corp",
    "inc", "co", "group", "holding", "holdings",
    "services", "service", "solutions", "enterprises",
    "international", "global", "partners", "associates",
    "consultancy", "consultants", "and", "india"
}

ADDRESS_STOPWORDS = {
    "road", "street", "avenue", "boulevard", "drive",
    "lane", "highway", "parkway", "court", "place",
    "trail", "circle", "suite", "apartment", "floor",
    "flat", "unit", "block", "plot", "sector", "nagar",
    "colony", "near", "opposite", "opp", "main", "city",
    "county", "north", "south", "east", "west", "the",
    "and", "for", "null"
}


def informative_name_tokens(text):
    tokens = normalize_name(text).split()
    return [
        t for t in tokens
        if len(t) >= 3 and t not in NAME_STOPWORDS
    ]


def informative_address_tokens(text):
    tokens = normalize_address(text).split()
    return [
        t for t in tokens
        if len(t) >= 3 and t not in ADDRESS_STOPWORDS
    ]

In [ ]:
name_index = defaultdict(set)
address_index = defaultdict(set)
name_token_temp = defaultdict(set)
address_pair_temp = defaultdict(set)

for entity_id, row in candidate_source.iterrows():

    country = str(row["country"]) if not pd.isna(row["country"]) else ""

    # Exact name
    name = normalize_name(row["business_name"])
    if name:
        name_index[(country, name)].add(entity_id)

    # Exact address
    address = normalize_address(row["business_address"])
    if address:
        address_index[(country, address)].add(entity_id)

    # Name tokens
    name_tokens = informative_name_tokens(row["business_name"])
    for token in set(name_tokens):
        name_token_temp[(country, token)].add(entity_id)

    # Address token pairs
    address_tokens = sorted(set(
        informative_address_tokens(row["business_address"])
    ))

    for a, b in combinations(address_tokens, 2):
        address_pair_temp[(country, a, b)].add(entity_id)

print("Name blocks:", len(name_index))
print("Address blocks:", len(address_index))
print("Name token blocks:", len(name_token_temp))
print("Address pair blocks:", len(address_pair_temp))